# Preparacion reproducible en Colab
Seleccionar runtime GPU T4 para entrenamiento/inferencia. Repositorio privado: cargar un ZIP autorizado mediante selector; no escribir tokens en celdas. Se trabaja desde raiz del proyecto. Las salidas vacias no constituyen evidencia de ejecucion.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/proyecto')
roots = list(Path('/content/proyecto').rglob('requirements.txt'))
assert len(roots) == 1, 'El ZIP debe contener una sola copia del proyecto'
os.chdir(roots[0].parent)
print(Path.cwd())

In [ ]:
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -r requirements.txt
!python -m pip check

In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes, datasets
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
print('transformers', transformers.__version__, 'PEFT', peft.__version__)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

# Evaluacion real
Ejecutar primero baseline, antes de entrenamiento. Configurar MySQL 8.4.10 de solo lectura y cargar esquemas/fixtures de validacion. Las preguntas necesitan revision humana y nunca deben formar parte de entrenamiento. Ejecutar la misma CLI y prompt para base, A y B, y despues estrategia directa/correccion. Ver README para comandos definitivos.

In [ ]:
!python -m scripts.bootstrap_mysql_colab
from scripts.with_env import load_env
load_env()
from src.data.mysql import MySQLExecutor, initialize
try:
    db = MySQLExecutor('hospital'); db.close()
except Exception:
    initialize()


In [ ]:
!python -m scripts.select_validation
!python -m scripts.with_env python -m scripts.evaluate --dataset datos/validacion/evaluacion_48.jsonl --config configs/baseline.json --strategy ambas


Tras baseline, ejecutar A y B usando las rutas adapters_experimentos/config_a y config_b. No interpretar una celda sin salida como experimento realizado. Guardar CSV por pregunta y resumenes reales. Promover pesos solamente despues de comparar EX y latencias.

Ejecutar las siguientes celdas **despues** del notebook 02, en runtime con los pesos reales.

In [ ]:
!python -m scripts.with_env python -m scripts.evaluate --dataset datos/validacion/evaluacion_48.jsonl --config configs/baseline.json --config configs/inference_a.json --config configs/inference_b.json --strategy ambas


In [ ]:
import shutil
shutil.make_archive('/content/evaluacion', 'zip', '.', 'resultados')
files.download('/content/evaluacion.zip')